In [1]:
import json
from dotenv import load_dotenv
from pandas import unique

from app.data.dto.main.SeaPort import SeaPortBubble

load_dotenv()

from app.services.db_service import DbService
sql_db_service = DbService()
await sql_db_service.init_pool()

from app.services.external_api.bubble_api import BubbleApi
api = BubbleApi("https://bunkering-backup-20251118-1137.bubbleapps.io/version-test/api/1.1/obj", "6a41c38bc132fcb7260656f3e9b916b3")

In [ ]:
all_ports_db, err = await sql_db_service.get_all_ports()

In [1]:

import json

with open("ports_json.json") as f:
    all_ports = json.loads(f.read())


In [10]:
wrong_lat_ports = [p for p in all_ports if p['sr_lat'] is not None and abs(p['sr_lat']) > 90]
countryes = list(set([p['sr_country_name'] for p in wrong_lat_ports]))

print("With LATITUDE > 90 count: ", len(wrong_lat_ports))
print("Countries with ports where LATITUDE > 90 count: ", len(countryes))
print(countryes)


With LATITUDE > 90 count:  92
Countries with ports where LATITUDE > 90 count:  10
['Indonesia', 'Australia', 'Japan', 'China', 'Viet Nam', 'Malaysia', 'Russian Federation', 'United States', 'Korea, Republic of', 'Papua New Guinea']


In [6]:
all_ports[0]

{'locode': 'ESAXO',
 'sr_lat': 43.3,
 'sr_country_name': 'Spain',
 'sr_lon': -8.5,
 'created_date': '2024-04-14T18:05:05.273Z',
 'created_by': 'admin_user_bunkering-53251_test',
 '_id': '1713117905273x207394668009318180',
 'search_key': 'a baiuca|esaxo|spain|',
 'modified_date': '2024-08-30T12:45:38.048Z',
 'name': 'A Baiuca',
 'mabux_ids': ['539']}

In [ ]:

for port in [p for p in all_ports if p['sr_lat'] is not None and (p['sr_lat'] > 90 or p['sr_lat'] < -90)]:
    port['sr_lat'], port['sr_lon'] =  port['sr_lon'], port['sr_lat']


inserted_index = []
failed_index = []

for i, port in enumerate(all_ports, 0):
    if port['locode'] is None:
        failed_index.append(i)

    port_db, err = await sql_db_service.get_port_by_locode(port['locode'])
    if port_db:
        continue

    if err == "Could not find port":

        new_port = SeaPortBubble(
            bubble_id=port['_id'],
            locode=port['locode'],
            country_name=port['sr_country_name'],
            port_name=port['name'],
            latitude=port['sr_lat'],
            longitude=port['sr_lon'],
            mobux_ids=port.get('mobux_ids',  []),
            search_key=port['search_key'],
        )

        new_port_bd, err = await sql_db_service.create_port_from_bubble(new_port)
        if new_port_bd:
            inserted_index.append(i)
        else:
            failed_index.append(i)


len(inserted_index), len(failed_index)

In [ ]:
[p for p in all_ports if p['locode'] == 'CNBLX']

In [ ]:
port = all_ports[0]
port.get('mobux_ids',  [])

#mobux_ids = json.loads()

In [ ]:
[p for p in all_ports if p['mobux_ids'] is None]

In [ ]:
len(all_ports)

In [ ]:
# import json
# ports_json = []
#
# for p in all_ports:
#     ports_json.append({
#      "locode": p.locode,
#         "sr_lat": p.sr_lat,
#         "sr_country_name": p.sr_country_name,
#         "sr_lon": p.sr_lon,
#         "created_date": str(p.created_date),
#         "created_by": p.created_by,
#         "_id": p.id,
#         "search_key": p.search_key,
#         "modified_date": p.modified_date,
#         "name": p.name,
#         "mobux_ids": p.mobux_ids,
#
#     })
#
# with open("ports_json.json", "w") as ports_json_file:
#     json.dump(ports_json, ports_json_file, indent=4)


In [2]:
all_ports = []
all_ports_raw = []

In [2]:
import json
with open("bubble_ports_raw.json", "r") as ports_json_file:
    all_ports_raw = json.load(ports_json_file)
len(all_ports_raw)


10297

In [4]:
[p for p in all_ports_raw if p.get('locode') == 'AEFJR']

[{'sr_lon': 56.3612937927246,
  'p: VLS FO': 565,
  '_id': '1713120159661x398876255901701800',
  'search_key': 'al fujayrah|aefjr|united arab emirates|3',
  'sr_country_name': 'United Arab Emirates',
  'Created By': 'admin_user_bunkering-53251_live',
  'sr_lat': 25.186993598938,
  'p: 380HSFO': 483,
  'p: MGO LS': 766,
  'by_truck': False,
  'locode': 'AEFJR',
  'average_agent_cost': 350,
  'bunkering_duration': 14,
  'Created Date': '2024-04-14T18:42:39.661Z',
  'name': 'Al Fujayrah',
  'by_barge': True,
  'is_shown': True,
  'latest_price_update': '2025-02-21T05:00:16.711Z',
  'agent_contact_list': 'agent required',
  'mabux_id': '3',
  'Modified Date': '2025-02-21T05:00:16.743Z'}]

In [ ]:
all_ports_raw[0]

In [5]:
[p for p in all_ports_raw if p.get('agent_contact_list') is not None]

[{'sr_country_name': 'Malta',
  'by_barge': False,
  'p: MGO LS': 700,
  'search_key': 'valletta|mtmla|malta|14',
  'is_shown': True,
  'mabux_id': '14',
  'Created Date': '2024-04-14T18:05:10.013Z',
  'source_mabux_ids': ['14'],
  'Created By': 'admin_user_bunkering-53251_test',
  'latest_price_update': '2024-09-11T03:46:11.058Z',
  'by_truck': False,
  'p: 180HSFO': 0,
  'p: VLS FO': 580,
  '_id': '1713117910013x878548882592774500',
  'Modified Date': '2024-10-02T14:44:41.990Z',
  'sr_lat': 35.8918285369873,
  'p: 380HSFO': 555,
  'agent_contact_list': 'Bunkering OPL exists',
  'sr_lon': 14.5096955299377,
  'name': 'Valletta',
  'manual_pricing': True,
  'locode': 'MTMLA'},
 {'Created Date': '2024-04-14T18:05:23.357Z',
  'agent_contact_list': 'BERTH and OUTER ANCHORAGE supply',
  'is_shown': True,
  'source_mabux_ids': ['299', '298', '229', '391'],
  'barging_cost': 2750,
  'name': 'Kandla',
  'mabux_id': '298',
  'manual_pricing': True,
  'p: MGO LS': 1035,
  'sr_country_name': 'Ind

In [ ]:
len([p for p in all_ports_raw if p.get("mabux_id") is not None])

In [4]:
async def process_port(port_dict, sql_db_service, semaphore):
    async with semaphore:
        try:
            # barge_status = port_dict.get("by_barge")
            # barge_status = bool(barge_status) if barge_status is not None else None
            #
            # truck_status = port_dict.get("by_truck")
            # truck_status = bool(truck_status) if truck_status is not None else None
            #
            # mabux_id = port_dict.get("mabux_id")
            # mabux_id = int(mabux_id) if mabux_id is not None else None

            agent_contact_list = port_dict.get('agent_contact_list')
            if not agent_contact_list:
                return

            locode = port_dict.get("locode")
            if not locode:
                return

            port_db, err = await sql_db_service.get_port_by_locode(locode)
            if not port_db:
                return

            fields = {
                "agent_contact_list": agent_contact_list,

            }

            _, err = await sql_db_service.upsert_mabux_fields(port_db.id, fields)

            if err:
                print(f"[{locode}] {err}")

        except Exception as e:
            print(f"[{port_dict.get('locode')}] unexpected error: {e}")

import asyncio

CONCURRENCY = 20  # should match or be slightly below pool max_size

semaphore = asyncio.Semaphore(CONCURRENCY)

tasks = [
    process_port(port_dict, sql_db_service, semaphore)
    for port_dict in all_ports_raw
]

await asyncio.gather(*tasks)

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [ ]:
# with open("bubble_ports_raw.json", "w") as fp:
#     json.dump(all_ports_raw, fp, indent=4)

In [ ]:
# import time
# import random
# current_cursor = 10000
#
# while True:
#     print(f"Fetching ports from cursor: {current_cursor}")
#
#     data, err = api.get_ports(obj_name="Port", offset=current_cursor)
#
#     if not data:
#         print("Failed to fetch data, stopping.")
#         print(err)
#         break
#
#     results = data.results
#
#     if not results:
#         print("No more results found.")
#         break
#
#     all_ports.extend(results)
#     all_ports_raw.extend(data.raw_results)
#
#     current_cursor = data.cursor + len(results)
#     remaining = data.remaining
#
#     print(f"Fetched {len(results)} ports. Total so far: {len(all_ports)}. Remaining: {remaining}")
#
#     if remaining <= 0:
#         print("Reached the end of all ports.")
#         break
#
#     time.sleep(random.uniform(2, 3))
